In [22]:
!pip install transformers datasets scikit-learn torch -q

In [23]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from tqdm import tqdm

In [24]:
from google.colab import files
uploaded = files.upload()

Saving classification_dataset.csv to classification_dataset (1).csv


In [25]:
df = pd.read_csv("classification_dataset.csv")

# Rename columns
df = df.rename(columns={"input": "text", "output": "label"})

# Convert labels (1–5 → 0–4)
df["label"] = df["label"] - 1

df.head()

,text,label
0,was zero or null on the most recent date in th...,2
1,combined data usage and outgoing call MOU over...,0
2,not subscribed to product in last 40 days,1
3,combined account balance and og call revenue o...,0
4,customers who activated HBB add-on yesterday,2


In [26]:
# Split the dataset: 80% for training the model, 20% for validating its performance.
# random_state ensures reproducibility (we get the same split every time we run it).

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
# Print the sizes of our splits to confirm.
print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 340
Validation size: 85


In [27]:
# Define the base model we want to fine-tune.

model_name = "answerdotai/ModernBERT-base"

# Download and load the Tokenizer associated with this model.
# The tokenizer converts our text sentences into numerical IDs that the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [28]:
# Create a custom PyTorch Dataset class to efficiently feed data to our model.

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
 # Returns the total number of items in the dataset.
    def __len__(self):
        return len(self.texts)

  # Tokenize the text on the fly.
  # It pads short sentences and truncates long ones to ensure a uniform length (max_len).

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
# Return the processed input tensors and the corresponding target label.
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [29]:
# Instantiate our custom dataset for both train and validation sets.

train_dataset = TextDataset(train_df["text"], train_df["label"], tokenizer)
val_dataset = TextDataset(val_df["text"], val_df["label"], tokenizer)

# Wrap datasets in DataLoaders to group data into batches of 16.
# We shuffle the training data so the model doesn't memorize the order of the inputs.
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [30]:
# Download the pre-trained ModernBERT model.
# We add a classification head on top with num_labels=6 to account for our 6 distinct tracks.

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
# Detect if a GPU (CUDA) is available in Colab for faster training; otherwise fallback to CPU.
# Move the model weights to the selected hardware device.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Using device:", device)

Using device: cuda


In [32]:

# Initialize the AdamW optimizer.
# A learning rate of 2e-5 is standard best practice for fine-tuning transformer models.

optimizer = AdamW(model.parameters(), lr=2e-5)

In [33]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")         # Initialize a progress bar for the current epoch.

    for batch in loop:
        optimizer.zero_grad()         # Clear old gradients from the last step.


# Move the batch data to our GPU/CPU.
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
 # Forward pass: get model predictions and calculate the loss (error).
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()
  # Backward pass: calculate the gradients (how much to adjust weights).
        loss.backward()

# Optimizer step: update the model's weights based on the gradients.
        optimizer.step()
 # Update the progress bar with the current loss.
        loop.set_postfix(loss=loss.item())
 # Print the average loss for the epoch to monitor learning progress.
    print(f"Epoch {epoch+1} Loss:", total_loss / len(train_loader))

Epoch 1: 100%|██████████| 22/22 [00:11<00:00,  1.90it/s, loss=0.687]


Epoch 1 Loss: 1.0177286551757292


Epoch 2: 100%|██████████| 22/22 [00:12<00:00,  1.80it/s, loss=0.129]


Epoch 2 Loss: 0.06613268449225208


Epoch 3: 100%|██████████| 22/22 [00:11<00:00,  1.88it/s, loss=0.0322]


Epoch 3 Loss: 0.020325766114348717


Epoch 4: 100%|██████████| 22/22 [00:11<00:00,  1.87it/s, loss=0.000885]


Epoch 4 Loss: 0.00234616953689097


Epoch 5: 100%|██████████| 22/22 [00:11<00:00,  1.88it/s, loss=0.000844]

Epoch 5 Loss: 0.0010643241548677906


In [34]:
model.eval()                  # Set the model to evaluation mode (disables training features like dropout).

correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.9882352941176471


In [40]:
from sklearn.metrics import classification_report                     # Lists to store our predictions and true labels for the classification report.

model.eval()         # Set the model to evaluation mode (disables training features like dropout).
all_preds = []       # Lists to store our predictions and true labels for the classification report.
all_labels = []

with torch.no_grad():   # torch.no_grad() disables gradient tracking to speed up evaluation and save memory.
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

      # Forward pass to get predictions.
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
 # Grab the highest probability class prediction.
        preds = torch.argmax(outputs.logits, dim=1)

        # Store predictions and true labels
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# This will print the Precision, Recall, and F1-Score for EACH of your 5 tracks!
print(classification_report(all_labels, all_preds, target_names=["Track 1", "Track 2", "Track 3", "Track 4", "Track 5"]))


              precision    recall  f1-score   support

     Track 1       1.00      0.95      0.97        19
     Track 2       1.00      1.00      1.00        18
     Track 3       1.00      1.00      1.00        12
     Track 4       0.94      1.00      0.97        16
     Track 5       1.00      1.00      1.00        20

    accuracy                           0.99        85
   macro avg       0.99      0.99      0.99        85
weighted avg       0.99      0.99      0.99        85



In [35]:
# a helper function to test the model with custom raw text. # Ensure model is in evaluation mode.

def predict(text):
    model.eval()

    inputs = tokenizer(                         # Tokenize the custom text snippet.
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():                            # Make the prediction.
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    return pred + 1  # back to original labels,  # Add 1 to map the 0-indexed prediction back to original Track numbers (1 to 5).


# Test
print(predict("customers who received a bonus for action key X in the last N days"))

5


In [38]:
# Save the final model weights and the tokenizer configuration locally.
# This allows us to load the model later for production deployment without retraining.

model.save_pretrained("bert_classifier")
tokenizer.save_pretrained("bert_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert_classifier/tokenizer_config.json', 'bert_classifier/tokenizer.json')